# VocEd Lab 05 — Convolutions & a Minimal CNN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/05_convolutions.ipynb)

In Lab 04 we classified pixels by their colour alone.  Each pixel was treated independently
— the classifier had no idea what pixels surrounded it.  In this lab we introduce the
**convolution operation**, which looks at a small *neighbourhood* (a patch) of pixels at
once.  This is the core building block of every modern image recognition system.

We start by applying filters manually so the maths is transparent, then we build a small
**Convolutional Neural Network (CNN)** in PyTorch that learns its own filter weights
directly from the training data.

By the end of this lab you will be able to:
- Compute a convolution by hand using a nested loop.
- Explain the difference between a blur kernel and an edge-detection kernel.
- Build a minimal PyTorch CNN (`nn.Module`) with `Conv2d`, `ReLU`, and a final 1×1 conv.
- Train the CNN on the training split and evaluate on the test split.
- Extend the cumulative Dice table with CNN results.

## 0. Setup

In [ ]:
# Clone the repo
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

## 1. Load Data & Recreate the Train/Test Split

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split

N = len(glob.glob('imagedata/X/*.npy'))
X = np.stack([np.load(f'imagedata/X/{i}.npy') for i in range(N)])
y = np.stack([np.load(f'imagedata/y/{i}.npy') for i in range(N)])

# Drop images with no nucleus pixels — identical to Lab 02
has_nucleus = (y == 2).sum(axis=(1, 2)) > 0
X, y = X[has_nucleus], y[has_nucleus]
N = len(X)

# Stratified split by N/C ratio — identical to Lab 02
nuc_px   = (y == 2).sum(axis=(1, 2))
cyt_px   = (y == 1).sum(axis=(1, 2))
nc_ratio = nuc_px / np.maximum(cyt_px, 1)   # np.maximum avoids div-by-zero
quartile = np.digitize(nc_ratio, np.percentile(nc_ratio, [25, 50, 75]))

train_idx, test_idx = train_test_split(
    np.arange(N), test_size=0.2, stratify=quartile, random_state=42
)

mask_cmap = ListedColormap(['black', 'steelblue', 'crimson'])
legend_patches = [
    mpatches.Patch(color='black',     label='0 — background'),
    mpatches.Patch(color='steelblue', label='1 — cytoplasm'),
    mpatches.Patch(color='crimson',   label='2 — nucleus'),
]
print(f'{N} images retained  ({(~has_nucleus).sum()} with no nucleus removed)')
print(f'Train: {len(train_idx)}   Test: {len(test_idx)}')

## 2. Convolution by Hand

A convolution slides a small **kernel** (a grid of weights) over every position in the
image.  At each position it multiplies the kernel weights with the pixel values underneath
and sums them up.  The result at that position is a single number: the **feature response**.

We work on the grayscale version of image 7 and try three kernels:
1. **Blur kernel** — all positive weights that sum to 1; averages the neighbourhood.
2. **Edge kernel** — detects horizontal edges; positive above, negative below.
3. **Random weights** — what does a kernel with no special structure produce?

Then we ask: *what if the network could learn the best weights by itself?*  That's the
leap to CNNs.

In [ ]:
# ── Manual 3×3 convolution (single-channel, no padding) ──────────────────────
def manual_conv2d(image, kernel):
    """
    Apply a 3×3 kernel to a 2-D image.  Returns an output smaller by 1 pixel
    on each side (no padding).

    image  : (H, W) float array
    kernel : (3, 3) float array
    """
    H, W = image.shape
    output = np.zeros((H - 2, W - 2))   # output shrinks because we need full 3×3 neighbourhoods

    for row in range(H - 2):
        for col in range(W - 2):
            # Extract the 3×3 patch centred at (row+1, col+1)
            patch = image[row : row + 3, col : col + 3]   # shape: (3, 3)
            # Element-wise multiply patch with kernel, then sum — this is a dot product
            output[row, col] = (patch * kernel).sum()

    return output


# Pick one image to demonstrate the kernels on
IDX  = 7
img  = X[IDX]
gray = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]   # (256, 256)

# ── Three kernels ─────────────────────────────────────────────────────────────
# Blur kernel: uniform average of the 3×3 neighbourhood
blur_kernel = np.ones((3, 3)) / 9.0

# Horizontal edge kernel: highlights rows where intensity changes sharply
# +1 on top, 0 in middle, -1 on bottom → large response at horizontal edges
edge_kernel = np.array([[ 1,  1,  1],
                         [ 0,  0,  0],
                         [-1, -1, -1]], dtype=float)

# Random kernel: random weights, no special structure
rng = np.random.default_rng(7)
rand_kernel = rng.standard_normal((3, 3))

# ── Apply and visualise ───────────────────────────────────────────────────────
# We use a 64×64 crop to keep the manual loop fast
crop = gray[96:160, 96:160]   # (64, 64) crop near the cell

print('Running manual convolutions on a 64×64 crop (may take a few seconds)...')
out_blur = manual_conv2d(crop, blur_kernel)
out_edge = manual_conv2d(crop, edge_kernel)
out_rand = manual_conv2d(crop, rand_kernel)
print('Done.')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

axes[0, 0].imshow(crop, cmap='gray');     axes[0, 0].set_title('Original crop');  axes[0, 0].axis('off')
axes[0, 1].imshow(out_blur, cmap='gray'); axes[0, 1].set_title('Blur kernel');    axes[0, 1].axis('off')
axes[0, 2].imshow(out_edge, cmap='RdBu_r', vmin=-0.5, vmax=0.5);
axes[0, 2].set_title('Edge kernel');  axes[0, 2].axis('off')
axes[0, 3].imshow(out_rand, cmap='RdBu_r');
axes[0, 3].set_title('Random kernel');  axes[0, 3].axis('off')

# Show the kernel weights themselves
for ax, k, title in zip(axes[1, :3],
                        [blur_kernel, edge_kernel, rand_kernel],
                        ['Blur', 'Edge', 'Random']):
    im = ax.imshow(k, cmap='RdBu_r')
    ax.set_title(f'{title} weights')
    plt.colorbar(im, ax=ax, shrink=0.8)

axes[1, 3].text(0.1, 0.5,
    'The CNN will\nlearn weights like\nthese automatically\nduring training.',
    transform=axes[1, 3].transAxes, fontsize=12, va='center')
axes[1, 3].axis('off')

plt.tight_layout()
plt.show()

## 3. Building a Minimal CNN in PyTorch

The manual loop above is far too slow for training.  PyTorch implements convolutions using
highly optimised GPU/CPU code.  More importantly, PyTorch can **differentiate through**
the convolution — it can compute how much the loss would change if each kernel weight
changed slightly.  This is the gradient we need to train the network.

Our minimal CNN:
```
Input (N, 3, 256, 256)
    → Conv2d(3, 16, 3, padding=1) + ReLU     # 16 filters of size 3×3
    → Conv2d(16, 32, 3, padding=1) + ReLU    # 32 filters of size 3×3
    → Conv2d(32, 3, 1)                        # 1×1 conv: maps 32 channels → 3 class logits
Output (N, 3, 256, 256)  ← one logit per class per pixel
```

`padding=1` keeps the spatial size the same (256×256 in, 256×256 out).  The final 1×1
convolution is just a per-pixel linear layer that mixes the 32 feature channels into 3
class scores.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Device selection: use GPU if available, otherwise CPU ─────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


# ── Dataset wrapper ───────────────────────────────────────────────────────────
# PyTorch requires a Dataset object that returns (image, label) pairs.
# __len__ returns the number of samples; __getitem__ returns one sample by index.
class SegDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = indices

    def __len__(self):
        return len(self.indices)   # how many images in this split

    def __getitem__(self, i):
        idx = self.indices[i]
        # torch.from_numpy wraps a NumPy array without copying data
        img  = torch.from_numpy(self.X[idx])   # (3, 256, 256) float32
        mask = torch.from_numpy(self.y[idx].astype(np.int64))  # (256, 256) int64
        return img, mask


# DataLoaders: shuffle training data each epoch; don't shuffle test data
train_loader = DataLoader(SegDataset(X, y, train_idx), batch_size=8, shuffle=True)
test_loader  = DataLoader(SegDataset(X, y, test_idx),  batch_size=8, shuffle=False)

print(f'Training batches: {len(train_loader)}   Test batches: {len(test_loader)}')

In [ ]:
# ── Minimal CNN ───────────────────────────────────────────────────────────────
# nn.Module is the base class for all PyTorch models.
# We define the layers in __init__ and describe the forward pass in forward().
class MinimalCNN(nn.Module):
    def __init__(self):
        super().__init__()   # always call super().__init__() first

        # Conv2d(in_channels, out_channels, kernel_size, padding)
        # padding=1 keeps spatial size the same when kernel_size=3
        self.conv1 = nn.Conv2d(3,  16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        # 1×1 convolution: acts as a per-pixel linear classifier
        # maps 32 feature channels → 3 class logits
        self.conv3 = nn.Conv2d(32,  3, kernel_size=1)

        self.relu = nn.ReLU()   # ReLU(x) = max(0, x) — adds non-linearity

    def forward(self, x):
        # x: (batch, 3, 256, 256)
        x = self.relu(self.conv1(x))   # → (batch, 16, 256, 256)
        x = self.relu(self.conv2(x))   # → (batch, 32, 256, 256)
        x = self.conv3(x)              # → (batch, 3, 256, 256) — logits, no activation
        return x


model = MinimalCNN().to(device)   # .to(device) moves the model to GPU if available
print(model)

# Count total trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

## 4. Training the CNN

In [ ]:
# ── Loss and optimiser ────────────────────────────────────────────────────────
# CrossEntropyLoss expects raw logits (no softmax) and integer labels.
# It is the standard loss for multi-class classification.
criterion = nn.CrossEntropyLoss()

# Adam is a popular gradient-descent optimiser.
# lr=1e-3 is the learning rate — how large each parameter update step is.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


# ── Training loop ─────────────────────────────────────────────────────────────
NUM_EPOCHS = 5   # 5 passes over the training data
train_losses = []

for epoch in range(NUM_EPOCHS):
    model.train()   # puts model in training mode (enables dropout, batch norm tracking, etc.)
    epoch_loss = 0.0

    for step, (imgs, masks) in enumerate(train_loader):
        imgs  = imgs.to(device)    # move batch to GPU/CPU
        masks = masks.to(device)

        # Forward pass: compute predictions
        logits = model(imgs)       # (batch, 3, 256, 256)

        # Compute loss: CrossEntropyLoss compares logits with ground-truth labels
        loss = criterion(logits, masks)

        # Backward pass: compute gradients
        optimizer.zero_grad()   # clear gradients from the previous batch
        loss.backward()         # backpropagate — fills .grad for every parameter

        # Update parameters: move each weight slightly in the direction that reduces loss
        optimizer.step()

        epoch_loss += loss.item()   # .item() converts a 1-element tensor to a Python float

        # Print per-batch loss every 5 batches so we can watch it bounce around
        if step % 5 == 0:
            print(f'  epoch {epoch+1}  batch {step:>3}/{len(train_loader)}  loss {loss.item():.4f}')

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}  —  avg loss: {avg_loss:.4f}')

# ── Plot training loss ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, NUM_EPOCHS + 1), train_losses, marker='o', color='steelblue')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Training Loss Curve — Minimal CNN')
plt.tight_layout()
plt.show()

## 5. Opening Up the Trained CNN

The CNN you just trained has 16 first-layer filters and 32 second-layer filters.  In §2
you saw what hand-designed kernels (Blur, Edge, Random) look like, and what they produce
when applied to an image.  Now we look at the **kernels the network learned by itself**
— and at the **feature responses** they produce on real cytology images.

Three questions to answer:

1. Do the learned kernels look like §2's hand-designed Blur or Edge filters?
2. Where in an image does each filter "fire"?
3. At depth 2, do channels start to specialise — and on what?

This section never trains the network again.  It only inspects the weights you already
have in `model`.


### 5.1 Conv1 — the 16 learned kernels

`model.conv1.weight` has shape `(16, 3, 3, 3)`: 16 filters, each a 3×3 kernel that
operates on 3 input channels (R, G, B).  Because there are exactly 3 input channels,
**each filter is itself a tiny 3×3 RGB image**.

Each kernel is normalised independently to `[0, 1]` for display — otherwise the
largest-weight filter dominates and the rest look black.


In [ ]:
W = model.conv1.weight.detach().cpu().numpy()    # (16, 3, 3, 3)

fig, axes = plt.subplots(4, 4, figsize=(7, 7))
for k, ax in enumerate(axes.flat):
    f = W[k].transpose(1, 2, 0)                          # (3, 3, 3) channel-last
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)       # normalise per filter
    ax.imshow(f)
    ax.set_title(f'filter {k}')
    ax.axis('off')
plt.suptitle('Conv1 — 16 learned 3×3 RGB kernels')
plt.tight_layout()
plt.show()


### 5.2 Conv1 feature maps — image 7

We ran image 7 through hand-designed Blur and Edge kernels in §2.  Now we run it through
all 16 **learned** conv1 kernels and look at the feature responses.

PyTorch lets us call any submodule of `model` directly: `model.conv1(image)` returns
the conv1 output without running the rest of the network.  We wrap with `torch.relu` to
get the post-ReLU activation that conv2 actually sees.

(Image 7 is a training image — the network has seen it.  That is fine for *"what did the
network learn?"* questions.  In §5.6 we will switch to test images when we rank channels.)


In [ ]:
model.eval()
with torch.no_grad():
    img_t = torch.from_numpy(X[7]).unsqueeze(0).to(device)   # (1, 3, 256, 256)
    a1 = torch.relu(model.conv1(img_t))                       # (1, 16, 256, 256)

A1 = a1.cpu().numpy()[0]   # (16, 256, 256)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for k, ax in enumerate(axes.flat):
    ax.imshow(A1[k], cmap='viridis')
    ax.set_title(f'channel {k}')
    ax.axis('off')
plt.suptitle('Conv1 feature maps for image 7 (post-ReLU)')
plt.tight_layout()
plt.show()


### 5.3 Conv1 across 4 images — does the response stay generic at depth 1?

Now we look at all 16 conv1 channels for **four images** spanning the full range of
N/C ratios:

| label | image | source | N/C ratio |
|---|---|---|---|
| **low N/C**     | 164 | test  | 0.03 |
| **image 7**     | 7   | train | 0.18 |
| **median N/C**  | 28  | test  | 0.43 |
| **high N/C**    | 151 | test  | 1.75 |

We expect channels at this shallow depth to respond fairly similarly across images —
edges and colour blobs are universal, regardless of how big the nucleus is.


In [ ]:
sample_idx = [164, 7, 28, 151]
sample_lbl = ['low N/C', 'image 7', 'median N/C', 'high N/C']

with torch.no_grad():
    imgs = torch.from_numpy(np.stack([X[i] for i in sample_idx])).to(device)
    A_4x16 = torch.relu(model.conv1(imgs)).cpu().numpy()    # (4, 16, 256, 256)

fig, axes = plt.subplots(4, 16, figsize=(20, 5))
for r in range(4):
    for c in range(16):
        axes[r, c].imshow(A_4x16[r, c], cmap='viridis')
        axes[r, c].set_xticks([])
        axes[r, c].set_yticks([])
    axes[r, 0].set_ylabel(sample_lbl[r])
plt.suptitle('Conv1 feature maps — 4 images × 16 channels')
plt.tight_layout()
plt.show()


### 5.4 Conv2 kernels — why we don't display them

Conv1's kernels were `(16, 3, 3, 3)` — 16 filters, each a 3×3 kernel with three colour
channels.  Tidy.

Conv2's kernels are `(32, 16, 3, 3)` — 32 filters, each with **16 input channels** that
come from conv1's outputs, not from RGB.  There is no longer a meaningful way to display
each filter as a single image.  We could plot 16 small heatmaps per filter
(32 × 16 = 512 panels — overwhelming), or some compressed summary that loses information.
Neither is useful.

This is a transferable insight: **past the first conv, kernel visualisation gets
impractical**.  The standard inspection tool for deeper layers is the **feature map**,
which is what we look at in §5.5.


### 5.5 Conv2 feature maps — image 7

Conv2 has 32 channels.  We compute its output by running conv1 first, then conv2:

`a2 = ReLU( conv2( ReLU( conv1(image) ) ) )`

We display all 32 channels for image 7 in a 4×8 grid.


In [ ]:
with torch.no_grad():
    img_t = torch.from_numpy(X[7]).unsqueeze(0).to(device)
    a1 = torch.relu(model.conv1(img_t))
    a2 = torch.relu(model.conv2(a1))                     # (1, 32, 256, 256)

A2 = a2.cpu().numpy()[0]   # (32, 256, 256)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for k, ax in enumerate(axes.flat):
    ax.imshow(A2[k], cmap='viridis')
    ax.set_title(f'channel {k}')
    ax.axis('off')
plt.suptitle('Conv2 feature maps for image 7 (post-ReLU)')
plt.tight_layout()
plt.show()


### 5.6 Choosing channels — Strategy 1 vs Strategy 2

There are 32 conv2 channels.  Showing all 32 across 4 images would be 128 panels — too
dense to read.  We need to pick a subset.  Here are two principled rankings.

**Strategy 1 — most active.**  For each channel, compute the mean activation magnitude
across a batch of test images.  Pick the top 6.  This shows the channels the network
*uses the most*.

**Strategy 2 — most nucleus-correlated.**  For each (channel, image) pair, compute the
**Pearson correlation** between the channel's activation map (256×256 floats) and the
**binary nucleus mask** `(y == 2)` for that image (256×256 zeros and ones).  Average
across images.  Pick the top 6.  This shows the channels whose firing pattern aligns
with where the nuclei actually are — the channels the network has internally allocated
to **nucleus detection**.

The two strategies usually give **different** top-6 lists.  A channel can fire loudly
everywhere (high S1, low S2).  A different channel can fire quietly but *precisely* on
nuclei (low S1, high S2).  The contrast between the two lists is the lesson.

Both rankings are computed on a batch of test images — no leakage from training data.


In [ ]:
# Held-out batch: all test images, conv2 activations + nucleus masks
batch_imgs  = torch.from_numpy(np.stack([X[i] for i in test_idx])).to(device)
batch_masks = (y[test_idx] == 2).astype(np.float32)            # (n_test, 256, 256)

model.eval()
with torch.no_grad():
    a1 = torch.relu(model.conv1(batch_imgs))
    a2 = torch.relu(model.conv2(a1)).cpu().numpy()             # (n_test, 32, 256, 256)

# ── Strategy 1: mean activation magnitude per channel ─────────────────────────
s1 = a2.mean(axis=(0, 2, 3))                                    # (32,)

# ── Strategy 2: mean Pearson correlation with nucleus mask ────────────────────
def pearson(a, b):
    a = a - a.mean(); b = b - b.mean()
    return (a * b).sum() / (np.sqrt((a*a).sum() * (b*b).sum()) + 1e-8)

s2 = np.zeros(32)
for c in range(32):
    s2[c] = np.mean([pearson(a2[i, c].ravel(), batch_masks[i].ravel())
                     for i in range(len(test_idx))])

top1 = np.argsort(s1)[::-1][:6]
top2 = np.argsort(s2)[::-1][:6]

print('Strategy 1 — most active channels:')
for c in top1:
    print(f'  channel {c:2d}   mean activation: {s1[c]:.3f}')

print('\nStrategy 2 — most nucleus-correlated channels:')
for c in top2:
    print(f'  channel {c:2d}   mean correlation: {s2[c]:+.3f}')

overlap = sorted(set(top1.tolist()) & set(top2.tolist()))
print(f'\nOverlap between top-6 lists: {len(overlap)} channel(s)  {overlap}')


### 5.7 The two top-6 grids — same 4 images, different channel sets

For each of the four sample images from §5.3, we plot its conv2 feature maps for the
top-6 channels under each strategy.

In the **Strategy 2** grid, look for activations that visibly trace the nuclei across
all four images — the network has dedicated these channels to nucleus detection.

In the **Strategy 1** grid, look for activations that are loud but more diffuse — these
are the channels the network uses heavily, even when they're not specifically about
nuclei.


In [ ]:
# Conv2 activations for the same 4 sample images
with torch.no_grad():
    sample_t  = torch.from_numpy(np.stack([X[i] for i in sample_idx])).to(device)
    sample_a1 = torch.relu(model.conv1(sample_t))
    sample_a2 = torch.relu(model.conv2(sample_a1)).cpu().numpy()    # (4, 32, 256, 256)

def show_grid(channels, title):
    fig, axes = plt.subplots(4, 6, figsize=(13, 9))
    for r in range(4):
        for j, c in enumerate(channels):
            axes[r, j].imshow(sample_a2[r, c], cmap='viridis')
            axes[r, j].set_xticks([])
            axes[r, j].set_yticks([])
        axes[r, 0].set_ylabel(sample_lbl[r])
    for j, c in enumerate(channels):
        axes[0, j].set_title(f'ch {c}')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_grid(top1, 'Strategy 1 — most active conv2 channels')
show_grid(top2, 'Strategy 2 — most nucleus-correlated conv2 channels')


### 5.8 Takeaways

- **Conv1 kernels look familiar.**  Some learned filters resemble §2's hand-designed
  Blur or Edge kernels.  Others don't have a clean interpretation — that is normal at
  this dataset size.
- **Depth induces specialisation.**  Conv1 channels (§5.2/§5.3) respond similarly
  across images — early features are generic.  Conv2 channels under Strategy 2 (§5.7)
  start to pick out the nucleus specifically.
- **"Most active" ≠ "most useful for the task."**  Strategy 1 and Strategy 2 typically
  rank channels differently.  A channel can fire loudly without being class-relevant.

In Project 3 you will do the same inspection on the U-Net, where the channel counts
are bigger (32 at enc1, 128 at the bottleneck) and the per-task specialisation is much
more pronounced.


## 6. Evaluation on the Test Set

In [ ]:
def dice_score(pred, target, cls):
    pred_mask   = (pred   == cls)
    target_mask = (target == cls)
    intersection = (pred_mask & target_mask).sum()
    denom = pred_mask.sum() + target_mask.sum()
    return 1.0 if denom == 0 else 2 * intersection / denom


model.eval()   # puts model in evaluation mode (disables dropout etc.)
cnn_scores = []

# torch.no_grad() tells PyTorch not to build a computation graph —
# we don't need gradients at test time, so this saves memory and speeds up inference.
with torch.no_grad():
    for imgs, masks in test_loader:
        imgs  = imgs.to(device)
        logits = model(imgs)                           # (batch, 3, 256, 256)
        preds  = logits.argmax(dim=1).cpu().numpy()    # take the class with the highest logit
        masks  = masks.numpy()

        for pred, mask in zip(preds, masks):
            d = (dice_score(pred, mask, 1) + dice_score(pred, mask, 2)) / 2
            cnn_scores.append(d)

mean_cnn = np.mean(cnn_scores)
print(f'Minimal CNN test Dice: {mean_cnn:.4f}  ±  {np.std(cnn_scores):.4f}')

In [ ]:
# ── Visual comparison: image 7 + 6 random test images ────────────────────────
def segment_plain(img, t_nucleus=0.45, t_background=0.85):
    gray = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nucleus]                                = 2
    pred[(gray >= t_nucleus) & (gray < t_background)]    = 1
    return pred

def segment_lab02(img):
    return segment_plain(img, t_nucleus=0.3904, t_background=0.9900)

# Image 7 on top, then 6 random test images (excluding 7 to avoid duplicates)
rng = np.random.default_rng(0)
candidates = np.array([i for i in test_idx if i != 7])
sample_indices = [7, *rng.choice(candidates, size=6, replace=False).tolist()]

model.eval()
fig, axes = plt.subplots(7, 5, figsize=(20, 28))
for row, idx in enumerate(sample_indices):
    img_t = torch.from_numpy(X[idx]).unsqueeze(0).to(device)
    with torch.no_grad():
        pred_cnn = model(img_t).argmax(dim=1).squeeze().cpu().numpy()
    pred_thresh = segment_plain(X[idx])
    pred_lab02  = segment_lab02(X[idx])

    axes[row, 0].imshow(X[idx].transpose(1, 2, 0));                                                     axes[row, 0].axis('off')
    axes[row, 1].imshow(y[idx],       cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest');         axes[row, 1].axis('off')
    axes[row, 2].imshow(pred_thresh,  cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest');         axes[row, 2].axis('off')
    axes[row, 3].imshow(pred_lab02,   cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest');         axes[row, 3].axis('off')
    axes[row, 4].imshow(pred_cnn,     cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest');         axes[row, 4].axis('off')

    if row == 0:
        axes[0, 0].set_title(f'RGB  (image {idx})')
        axes[0, 1].set_title('Ground truth')
        axes[0, 2].set_title('Lab 01 threshold')
        axes[0, 3].set_title('Lab 02 — Bayesian opt')
        axes[0, 4].set_title('Lab 05 — Minimal CNN')
    else:
        axes[row, 0].set_title(f'image {idx}')

fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, 0.0))
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()

# Cumulative Dice table
d_lab01 = np.mean([
    (dice_score(segment_plain(X[i]), y[i], 1) + dice_score(segment_plain(X[i]), y[i], 2)) / 2
    for i in test_idx
])
d_lab02 = np.mean([
    (dice_score(segment_lab02(X[i]), y[i], 1) + dice_score(segment_lab02(X[i]), y[i], 2)) / 2
    for i in test_idx
])
print('\nCumulative Dice comparison (test set):')
print('=' * 50)
print(f'{"Lab 01 — hand-picked thresholds":<35}  {d_lab01:.4f}')
print(f'{"Lab 02 — Bayesian optimisation":<35}  {d_lab02:.4f}')
print(f'{"Lab 05 — Minimal CNN":<35}  {mean_cnn:.4f}')
print('=' * 50)

## 7. N/C Ratio Scatter — Predicted vs True

Dice measures mask overlap.  Here we check whether that improvement in Dice actually
translates into a more accurate **N/C ratio** — the quantity a pathologist would measure.
Points near the y = x line mean the predicted ratio closely matches the ground truth.

In [ ]:
def r2_identity(yp, gt):
    """R² vs y = x: 1 means perfect agreement with the identity line."""
    ss_res = np.sum((yp - gt) ** 2)
    ss_tot = np.sum((gt - gt.mean()) ** 2)
    return 1 - ss_res / ss_tot

def nc_ratio(mask):
    """N/C ratio: nucleus pixel count / cytoplasm pixel count. Returns nan if no cytoplasm."""
    nuc = (mask == 2).sum()
    cyt = (mask == 1).sum()
    return nuc / cyt if cyt > 0 else np.nan


# ── True N/C ratios from ground-truth masks ───────────────────────────────────
nc_true = np.array([nc_ratio(y[i]) for i in test_idx])

# ── Lab 01: hand-picked threshold predictions ─────────────────────────────────
nc_lab01 = np.array([nc_ratio(segment_plain(X[i])) for i in test_idx])

# ── Lab 02: Bayesian-optimised threshold predictions ─────────────────────────
nc_lab02 = np.array([nc_ratio(segment_lab02(X[i])) for i in test_idx])

# ── CNN: run inference one image at a time ────────────────────────────────────
model.eval()
nc_cnn = []
with torch.no_grad():
    for i in test_idx:
        img_t = torch.from_numpy(X[i]).unsqueeze(0).to(device)
        pred  = model(img_t).argmax(dim=1).squeeze().cpu().numpy()
        nc_cnn.append(nc_ratio(pred))
nc_cnn = np.array(nc_cnn)

# ── Scatter plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, nc_pred, label, color in zip(
        axes,
        [nc_lab01, nc_lab02, nc_cnn],
        ['Lab 01 — hand-picked', 'Lab 02 — Bayesian opt', 'Lab 05 — Minimal CNN'],
        ['steelblue', 'seagreen', 'crimson']):

    valid = np.isfinite(nc_true) & np.isfinite(nc_pred)
    x, yp = nc_true[valid], nc_pred[valid]

    r2 = r2_identity(yp, x)

    ax.scatter(x, yp, alpha=0.6, color=color, edgecolors='none', s=40)
    lim = max(x.max(), yp.max()) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1.5, label='y = x (perfect)')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_aspect('equal')
    ax.set_xlabel('True N/C ratio')
    ax.set_ylabel('Predicted N/C ratio')
    ax.set_title(f'{label}\nR² = {r2:.3f}')
    ax.legend(fontsize=9)

plt.suptitle('N/C Ratio: Predicted vs True (test set)', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Residual Analysis: False Negatives and False Positives

The scatter above tells us how the CNN does on average; it does not tell us *which*
images it gets badly wrong.  The **residual** is `predicted − true`:

- **Strongly negative residual** — points well below the `y = x` line.  The model
  under-predicts the N/C ratio: a high-N/C cell is reported as low-N/C.  Clinically a
  **false negative** — the most dangerous failure mode for a cancer screen, since a
  positive case would be missed.
- **Strongly positive residual** — points well above `y = x`.  The model over-predicts:
  a low-N/C (healthy) cell is reported as high-N/C.  Clinically a **false positive** —
  unnecessary follow-up but not a missed diagnosis.

We rank every test image with a finite N/C ratio by its signed residual, then visualise
the four worst on each side: original RGB, ground-truth mask, CNN prediction, and the
rank.

When examining these cases, ask: *does the ground-truth annotation itself look right?*
If the cytopathologist's mask shows a tiny nucleus but the cell visually has a large
dense one (or the reverse), the residual is diagnosing a **labelling error** rather
than a model error.  Cross-checking like this is routine in medical-imaging projects.

In [ ]:
# ── Compute signed residuals on test images with valid N/C ratios ────────────
valid = np.isfinite(nc_true) & np.isfinite(nc_cnn)
test_valid = np.array(test_idx)[valid]      # absolute image indices
true_v = nc_true[valid]
pred_v = nc_cnn[valid]
res    = pred_v - true_v                    # residual: predicted - true

order   = np.argsort(res)                   # ascending: most negative first
N_valid = len(order)

worst_neg = order[:4]                       # 4 worst false negatives
worst_pos = order[-4:][::-1]                # 4 worst false positives, biggest first


def residual_grid(rel_indices, title, side_label):
    fig, axes = plt.subplots(4, 3, figsize=(11, 16))
    for row, ri in enumerate(rel_indices):
        abs_idx = test_valid[ri]
        img_t = torch.from_numpy(X[abs_idx]).unsqueeze(0).to(device)
        with torch.no_grad():
            pred_mask = model(img_t).argmax(dim=1).squeeze().cpu().numpy()

        rank = int(np.where(order == ri)[0][0]) + 1   # 1-indexed rank by ascending residual

        axes[row, 0].imshow(X[abs_idx].transpose(1, 2, 0))
        axes[row, 0].set_title(f'image {abs_idx}')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(y[abs_idx], cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
        axes[row, 1].set_title(f'GT mask   true N/C = {true_v[ri]:.3f}')
        axes[row, 1].axis('off')

        axes[row, 2].imshow(pred_mask, cmap=mask_cmap, vmin=0, vmax=2, interpolation='nearest')
        axes[row, 2].set_title(
            f'CNN pred   pred N/C = {pred_v[ri]:.3f}\n'
            f'residual = {res[ri]:+.3f}   rank {rank}/{N_valid} {side_label}')
        axes[row, 2].axis('off')

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


residual_grid(worst_neg,
              'Worst FALSE NEGATIVES — predicted N/C far too LOW (cancer indicator missed)',
              '(most negative)')
residual_grid(worst_pos,
              'Worst FALSE POSITIVES — predicted N/C far too HIGH (healthy cell flagged)',
              '(most positive)')


# ── Full residual ranking table ──────────────────────────────────────────────
print('Full residual ranking (signed: predicted - true, ascending):')
print(f'{"rank":>4}  {"image":>5}  {"true N/C":>9}  {"pred N/C":>9}  {"residual":>9}')
print('-' * 50)
for r, ri in enumerate(order):
    abs_idx = test_valid[ri]
    print(f'{r+1:>4}  {abs_idx:>5}  {true_v[ri]:>9.3f}  {pred_v[ri]:>9.3f}  {res[ri]:>+9.3f}')

## Wrap-up

**Key takeaways:**
- A convolution looks at a *patch* — it incorporates information from neighbouring pixels,
  which is more powerful than classifying each pixel in isolation.
- The CNN learns its filter weights from the data, so it automatically finds the
  patterns most useful for distinguishing the three classes.
- But our minimal CNN never changes its spatial resolution: every layer sees the same
  256×256 grid.  It has **no ability to zoom out** and see larger structures (e.g., the
  full shape of a cell).  This is the bottleneck we fix in Lab 06 with a U-Net.

---

## Group Exercise — Filter Depth

The number of filters per layer controls how many distinct patterns the CNN can detect.

| Person | Architecture to test |
|---|---|
| A | `Conv2d(3,8,3)` → `Conv2d(8,8,1)` (very shallow) |
| B | `Conv2d(3,16,3)` → `Conv2d(16,32,3)` → `Conv2d(32,3,1)` (this lab's default) |
| C | `Conv2d(3,32,3)` → `Conv2d(32,64,3)` → `Conv2d(64,3,1)` (wider) |

Train each architecture for 5 epochs and record:
- Final training loss.
- Dice on `test_idx[0:5]`.
- Number of trainable parameters.

Discuss: does more depth/width always help?  What is the trade-off?